# Combined exact-Hessian CUDA A/B

This notebook measures the first post-graph optimization for issue #122: replacing separate Gauss–Newton, residual-curvature, and constraint-curvature work with one exact Hessian of the per-segment Lagrangian.

It runs both implementations in the **same Colab session**:

- `legacy`: two nested Hessian transforms plus separately assembled Gauss–Newton terms.
- `combined`: one nested Hessian transform that directly differentiates the complete Lagrangian.

The quick test captures each real IPOPT Hessian callback on a one-day problem, evaluates both at identical inputs, and times repeated calls. The final cell optionally runs a longer end-to-end comparison.

Use **Runtime → Change runtime type → GPU**, then **Run all**. An A100 is preferred because this workload uses FP64.

In [ ]:
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore", message="Failed to parse namespace")
warnings.filterwarnings("ignore", message="Failed to parse ontology namespace")
warnings.filterwarnings("ignore", message='Neither "df", "filename", nor "uuid"')

TWIN4BUILD_REF = "feature/issue-122/cuda-graph-hessian"

# Always install the requested ref so a preinstalled release cannot be measured
# accidentally. In a fresh Colab runtime Twin4Build has not yet been imported,
# so the newly installed package is loaded immediately below.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
     f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
    check=True,
)
import twin4build as tb
import twin4build.estimator._transcription as _tr

if "TWIN4BUILD_COMBINED_HESSIAN" not in open(_tr.__file__, encoding="utf-8").read():
    raise RuntimeError(
        "Installed ref lacks the A/B switch. Disconnect and delete the Colab "
        "runtime, then Run all; a previously imported module is still cached."
    )

import functools
import os
import platform
import time

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU required: Runtime > Change runtime type > GPU")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor})")
print("CPU:", platform.processor() or platform.machine(), "cores:", os.cpu_count())
print("Torch:", torch.__version__)
print("Twin4Build:", tb.__file__)
print("Ref:", TWIN4BUILD_REF)

# CUDA graphs are deliberately disabled: this notebook measures eager Hessian
# organization, and failed graph capture would poison the CUDA context.
os.environ["TWIN4BUILD_CUDA_GRAPH"] = "0"
os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = "1"

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples as _ex_pkg
import twin4build.examples.utils as utils

# The data package shadows full_workflow_example.py, so load the module by path.
_path = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_combined_hessian", _path)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
N_WARMUP = 20
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
QUICK_END = [datetime.datetime(2023, 12, 3, tzinfo=tz.gettz("Europe/Copenhagen"))]
FULL_END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]


def build_model(tag):
    model = tb.Model(id=tag)
    model.load(
        semantic_model_filename=utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


BASE_OPTIONS = {
    "boundary_state_init": "rollout",
    "early_stopping": False,
    "exact_hessian": True,
}
print("Model builders ready")

In [ ]:
import twin4build.estimator._casadi_ipopt as _ipopt

CAPTURED = {}


def capture_arm(label, combined, maxiter=2):
    """Build the real callback and retain its first real IPOPT arguments."""
    os.environ["TWIN4BUILD_COMBINED_HESSIAN"] = "1" if combined else "0"
    original = _ipopt.solve_ipopt_constrained

    def wrapped(
        x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc,
        options=None, *, hess_vals=None, **kwargs
    ):
        @functools.wraps(hess_vals)
        def record(*args):
            if "args" not in CAPTURED[label]:
                CAPTURED[label]["args"] = tuple(
                    np.asarray(a).copy() if hasattr(a, "__len__") else a
                    for a in args
                )
            return hess_vals(*args)

        CAPTURED[label]["fn"] = hess_vals
        return original(
            x0, lb, ub, fun, grad, n_g, g_fun, g_jac_vals, jr, jc, options,
            hess_vals=record, **kwargs
        )

    CAPTURED[label] = {}
    _ipopt.solve_ipopt_constrained = wrapped
    try:
        model = build_model(f"capture_{label}")
        estimator = tb.Estimator(tb.Simulator(model))
        options = dict(BASE_OPTIONS, maxiter=maxiter)
        estimator.estimate(
            START, QUICK_END, STEP,
            build_parameters(model), build_measurements(model),
            n_warmup=N_WARMUP,
            method=("casadi", "ipopt", "ad", "collocation"),
            options=options,
        )
    finally:
        _ipopt.solve_ipopt_constrained = original


for label, combined in (("legacy", False), ("combined", True)):
    print(f"Capturing {label} callback ...", flush=True)
    capture_arm(label, combined)

# Evaluate both closures at exactly the same primal point, objective scale, and
# constraint multipliers. This is stronger than comparing two solver endpoints.
common_args = CAPTURED["combined"]["args"]
outputs = {}
rows = []
for label in ("legacy", "combined"):
    fn = CAPTURED[label]["fn"]
    outputs[label] = fn(*common_args)
    torch.cuda.synchronize()
    samples = []
    for _ in range(3):
        start = time.perf_counter()
        fn(*common_args)
        torch.cuda.synchronize()
        samples.append(time.perf_counter() - start)
    rows.append({
        "arm": label,
        "median_callback_s": float(np.median(samples)),
        "min_callback_s": float(np.min(samples)),
        "samples_s": [round(x, 4) for x in samples],
    })

legacy = outputs["legacy"]
combined = outputs["combined"]
delta = np.abs(combined - legacy)
scale = max(float(np.max(np.abs(legacy))), 1e-12)
print("\nNumerical equivalence at identical inputs")
print("  values:", len(legacy))
print("  max |delta|:", float(delta.max()))
print("  relative max delta:", float(delta.max() / scale))
print("  allclose (rtol=1e-9, atol=1e-10):", np.allclose(combined, legacy, rtol=1e-9, atol=1e-10))

callback_results = pd.DataFrame(rows)
base = callback_results.loc[callback_results.arm == "legacy", "median_callback_s"].iloc[0]
new = callback_results.loc[callback_results.arm == "combined", "median_callback_s"].iloc[0]
callback_results["speedup_vs_legacy"] = base / callback_results["median_callback_s"]
display(callback_results)
print(f"Combined callback speedup: {base / new:.2f}x")

## Optional end-to-end A/B

The callback comparison above isolates the optimization. The next cell measures whether it survives IPOPT overhead on the full five-day problem. It runs 100 iterations per arm and may take 10–20 minutes total depending on the GPU.

Endpoint fits can differ slightly because changing floating-point operation order changes IPOPT's trajectory. Treat callback speed and exact-input equivalence as the primary result; use the audit values below to detect a material convergence regression.

In [ ]:
RUN_END_TO_END = True
END_TO_END_ITERS = 100

end_to_end_rows = []
if RUN_END_TO_END:
    for label, combined_flag in (("legacy", False), ("combined", True)):
        print(f"\nRunning full-horizon {label} arm ...", flush=True)
        os.environ["TWIN4BUILD_COMBINED_HESSIAN"] = "1" if combined_flag else "0"
        model = build_model(f"e2e_{label}")
        estimator = tb.Estimator(tb.Simulator(model))
        options = dict(BASE_OPTIONS, maxiter=END_TO_END_ITERS)
        started = time.perf_counter()
        result = estimator.estimate(
            START, FULL_END, STEP,
            build_parameters(model), build_measurements(model),
            n_warmup=N_WARMUP,
            method=("casadi", "ipopt", "ad", "collocation"),
            options=options,
        )
        elapsed = time.perf_counter() - started
        audit = result.get("transcription_audit", {})
        per_sensor = audit.get("per_sensor", {})
        end_to_end_rows.append({
            "arm": label,
            "seconds": elapsed,
            "max_defect": audit.get("max_abs_defect", np.nan),
            "sensor_rmse": {
                key: round(float(value["nlp_rmse"]), 6)
                for key, value in per_sensor.items()
            },
        })
        del estimator, model
        torch.cuda.empty_cache()

    end_to_end_results = pd.DataFrame(end_to_end_rows)
    old = end_to_end_results.loc[
        end_to_end_results.arm == "legacy", "seconds"
    ].iloc[0]
    end_to_end_results["speedup_vs_legacy"] = (
        old / end_to_end_results["seconds"]
    )
    display(end_to_end_results)
else:
    print("Skipped. Set RUN_END_TO_END=True and rerun this cell when desired.")